In [1]:
from pathlib import Path
from pprint import pprint
from plant_pheno.inference import inat_client
from plant_pheno.data import DuckDbSQL, DuckDBAdapter
from plant_pheno.inference.inat_client import BinaryFetcher, LocalBinaryWriter
from plant_pheno.config import IngestPhotosParams


from typing import Dict, List

import aiohttp
from tqdm.asyncio import tqdm_asyncio

2026/09/18 13:08:19 INFO mlflow.models.signature: Unsupported type hint: pd.DataFrame, skipping schema inference
/home/etienne/miniconda3/envs/inat_cv/lib/python3.10/site-packages/mlflow/pyfunc/utils/data_validation.py:187: UserWarning: Add type hints to the `predict` method to enable data validation and automatic signature inference during model logging. Check https://mlflow.org/docs/latest/model/python_model.html#type-hint-usage-in-pythonmodel for more details.
  color_warning(


In [2]:

obs_id = [182806784,398318023]
rate = 10

TARGET_TABLE_NAME = "raw.inat_api"
SOURCE_KEY = 'uuid'
OBSERVATIONS_FIELDS = {
    "id" : True,
    "photos" : True,
}

DB_path = "/home/etienne/projects/inat-phenology-cv/data/cv_raw.duckdb"
SQL_api_path = Path("/home/etienne/projects/inat-phenology-cv/queries/api/")

In [3]:
with DuckDBAdapter(DB_path) as con:
    sql_api = DuckDbSQL(con, SQL_api_path)
    sql_api.execute("create_api_raw_table", table_name=TARGET_TABLE_NAME)

    
    config = inat_client.EndpointConfig(
        "observations",
        write_empty_rows=True,
        fields = OBSERVATIONS_FIELDS,
        chunk_size= 200,
        per_page= 200
    )



    fetcher = inat_client.fetchers.RateLimiterFetcher(rate = 10, ignore_not_found= True)
    with inat_client.DuckDbWriter(con, TARGET_TABLE_NAME ) as writer:
        client = inat_client.make_client(config, fetcher, writer)
        x = await client.execute(obs_id)

    sql_stage = DuckDbSQL(con, Path("/home/etienne/projects/inat-phenology-cv/queries/stage"))
    sql_stage.execute("stage_inat_requests")
    sql_stage.execute("stage_obs_photos")



post


100%|██████████| 1/1 [00:00<00:00,  1.96it/s]


In [ ]:

async def _download_photo(
    session: aiohttp.ClientSession,
    fetcher: BinaryFetcher,
    writer: LocalBinaryWriter,
    item_id: str,
    params: IngestPhotosParams

):

    """Orchestrate the download and write of a single photo."""
    extension = params.extension
    size = params.size


    url = f"https://inaturalist-open-data.s3.amazonaws.com/photos/{item_id}/{size}.{extension}"
    print(url)
    filename = f"{item_id}.{extension}"

    try:
        data = await fetcher.fetch(session, url)
        await writer.write(data, filename)
    except Exception as e:
        print("Failed to download photo %s: %s", item_id, e)


async def execute_async(
    items: List[Dict], target_dir: str, rate: int, params: IngestPhotosParams
):
    """Async execution of the photo download batch."""
    fetcher = BinaryFetcher(rate=rate)
    writer = LocalBinaryWriter(target_dir)

    async with aiohttp.ClientSession() as session:
        tasks = [
            _download_photo(
                session = session,
                fetcher = fetcher,
                writer = writer,
                item_id = str(item[params.item_id]),
                params = params
            )
            for item in items
        ]
        await tqdm_asyncio.gather(*tasks)

    writer.close()


In [12]:

SOURCE_TABLE_NAME = "staged.inat_request_photos"
PHOTO_FOLDER = "/home/etienne/projects/inat-phenology-cv/data/test_images"

params = IngestPhotosParams()
with DuckDBAdapter(DB_path) as con:
    sql_api = DuckDbSQL(con, SQL_api_path)
    # 2 Get missing items not collected
    df = sql_api.fetch_df_query(f"SELECT * FROM {SOURCE_TABLE_NAME}")
    # Convert rows to list of dicts
    items = df.to_dict(orient="records")

    print(items)

    print("Starting photo download for %d items", len(items))
    await (execute_async(items, PHOTO_FOLDER, rate, params))

print("Photo download complete.")



[{'observation_id': 182806784, 'photo_id': 318807114}, {'observation_id': 182806784, 'photo_id': 318807133}, {'observation_id': 182806784, 'photo_id': 318807187}, {'observation_id': 398318023, 'photo_id': 730674022}, {'observation_id': 398318023, 'photo_id': 730699267}]
Starting photo download for %d items 5


Request error on https://inaturalist-open-data.s3.amazonaws.com/photos/318807187/medium.jpg attempt 0 — waiting 2.2s: 404, message='Not Found', url='https://inaturalist-open-data.s3.amazonaws.com/photos/318807187/medium.jpg'
Request error on https://inaturalist-open-data.s3.amazonaws.com/photos/318807114/medium.jpg attempt 0 — waiting 3.9s: 404, message='Not Found', url='https://inaturalist-open-data.s3.amazonaws.com/photos/318807114/medium.jpg'
Request error on https://inaturalist-open-data.s3.amazonaws.com/photos/318807133/medium.jpg attempt 0 — waiting 2.9s: 404, message='Not Found', url='https://inaturalist-open-data.s3.amazonaws.com/photos/318807133/medium.jpg'


Request error on https://inaturalist-open-data.s3.amazonaws.com/photos/318807187/medium.jpg attempt 1 — waiting 4.2s: 404, message='Not Found', url='https://inaturalist-open-data.s3.amazonaws.com/photos/318807187/medium.jpg'
Request error on https://inaturalist-open-data.s3.amazonaws.com/photos/318807133/medium.jpg attemp

CancelledError: 

Failed to download photo %s: %s 318807187 Failed after 3 retries for https://inaturalist-open-data.s3.amazonaws.com/photos/318807187/medium.jpg
Failed to download photo %s: %s 318807133 Session is closed
Failed to download photo %s: %s 318807114 Session is closed
